In [1]:
import numpy as np
import math
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
Nx = 200
cfl = 0.2
tmax = 1
viscosity_coeff = (0.01/math.pi)

x = np.linspace(-1,1,200)
u = -1 * np.sin(math.pi * x)

dx = abs(x[1] - x[0])
dt = cfl * dx / np.max(np.abs(u))

nt = int(tmax / dt) + 1

uf = np.zeros((nt, Nx))
uf[0, :] = u

print(nt)
print(dt)
print(nt * dt, tmax)

498
0.0020101128725153656
1.001036210512652 1


In [5]:
def f(u):
    y = 0.5 * u**2
    yp = u 
    return y, yp

In [6]:
def minmod(a, b):
    return 0.5 * (np.sign(a) + np.sign(b)) * np.minimum(np.abs(a), np.abs(b))

In [14]:
def RHS(u, dx, viscosity_coeff):
    diffusion_term = viscosity_coeff * (np.roll(u,1) - 2 * u + np.roll(u,-1)) / dx**2

    ux = minmod((u - np.roll(u,1)) / dx, (np.roll(u,-1)))

    u_L = np.roll(u -0.5* dx*ux, 1)
    u_R = u - 0.5 * dx * ux
    
    f_L, fp_L = f(u_L)
    f_R, fp_R = f(u_R)

    a = np.maximum(np.abs(fp_L), np.abs(fp_R))

    H = 0.5 * (f_L + f_R - a * (u_R - u_L))
    conv_term = -(np.roll(H, -1) - H) / dx
    y = conv_term * diffusion_term
    return y

In [16]:
for i in range(1, nt):
    u1 = u + dt * RHS(u=u, dx=dx, viscosity_coeff=viscosity_coeff)
    u = 0.5 * u + 0.5 * (u1 + dt * RHS(u=u1, dx=dx, viscosity_coeff=viscosity_coeff))
    uf[i, :] = u